In [1]:
import random

from rdkit import Chem

from stereomolgraph import AtomId, Bond, MolGraph
from stereomolgraph.experimental.canon import canon_atom_num

In [2]:
def shuffle_graph(g: MolGraph) -> MolGraph:
    random_atom_ids: list[AtomId] = random.sample(range(len(g)), len(g))

    old_new_mapping: dict[AtomId, AtomId] = {}
    new_old_mapping: dict[AtomId, AtomId] = {}

    new_g = g.__class__()

    for atom, atom_type in random.sample(list(zip(g.atoms, g.atom_types)), len(g)):
        new_atom_id = random_atom_ids.pop()
        old_new_mapping[atom] = new_atom_id
        new_old_mapping[new_atom_id] = atom
        new_g.add_atom(old_new_mapping[atom], atom_type)

    for a1, a2 in random.sample(list(g.bonds), len(g.bonds)):
        new_g.add_bond(old_new_mapping[a1], old_new_mapping[a2])

    return new_g


In [3]:
def graph_to_set(g: MolGraph) -> frozenset[tuple[AtomId, int] | Bond]:
    ret: set[tuple[AtomId, int] | Bond] = {
        (a, t) for a, t in zip(g.atoms, g.atom_types)
    }
    for bond in g.bonds:
        ret.add(bond)
    return frozenset(ret)

In [4]:
def smiles2graph(smiles: str) -> MolGraph:
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    # mg = MolGraph.from_rdmol(rdmol)
    smg = MolGraph.from_rdmol(rdmol)
    return smg

In [5]:
def test_canon_atom_num(g: MolGraph, n: int = 10) -> bool:
    unique_graphs = set()
    for _ in range(n):
        shuffeled_g = shuffle_graph(g)
        canon_id_mapping = canon_atom_num(shuffeled_g)
        canon_graph = shuffeled_g.relabel_atoms(canon_id_mapping)
        unique_graphs.add(graph_to_set(canon_graph))
        assert {len(ug) for ug in unique_graphs} == {len(g.atoms) + len(g.bonds)}
    return True if len(unique_graphs) == 1 else False

In [6]:
difficult_graphs = (
    # "C1OC23COC45COC11COC67COC8(COC9(CO2)COC(CO1)(CO6)OCC(CO9)(OC4)OCC(CO5)(OC7)OC8)OC3",
    # # this is crazy slow because of a lot "equivalent atoms"
    "C12C3C4C5C1C6C7C2C8C3C6C5C8C74",  # peterson graph G(7,2)
    "C1(C2C3C4C15)C6C7C2C8C3C9C%10C4C%11C5C6C%12C%11C%10C%13C%12C7C8C9%13",
    "C1CC2CCC1CCC3CCC(CC3)CC2",
    "C12C3C1C4C5C4C5C23",
    "C1C2CC3CC1CC(C2)C3",  # adamantane
    "C12C3C4C1C5C2C3C45",  # cubane
)

Examples from: 
Krotko, D.G. Atomic ring invariant and Modified CANON extended connectivity algorithm for symmetry perception in molecular graphs and rigorous canonicalization of SMILES. J Cheminform 12, 48 (2020). https://doi.org/10.1186/s13321-020-00453-4

In [7]:
for smiles in difficult_graphs:
    g = smiles2graph(smiles)
    assert test_canon_atom_num(g), smiles